In [ ]:
# Normal:     0928
# IP, DeviceId, UserAgent
# 111.90.238.224, 435BB410-6433-453A-95C0-590E78543A64 , openrice_ios/7.21.0 (iphone; ios 18.5)
# 112.198.104.210, 72dd0c61450f9efd, openrice_android/7.21.0 (android; 35) [cph2799]
# 136.158.43.30, 8503e3f7ea531cf4, openrice_android/7.21.0 (android; 36) [sm-a566b]

# Abnormal (from Pakistan):
# 223.123.7.182,  cc21b7620eaf4e65, openrice_android/7.20.2 (android; 31)
# 223.123.7.182,  c9a76abc2baaf940,  openrice_android/7.20.2 (android; 31)
# 119.155.211.76, 907f26030cdd51e1, openrice_android/7.20.4 (android; 33) [infinix x6836]

In [2]:
import datetime 
from dateutil.relativedelta import relativedelta
import pandas as pd
import numpy as np
import db_dtypes
import pymssql
from shutil import copyfile
from openpyxl import load_workbook
import os
from google.cloud import bigquery
os.environ['GOOGLE_APPLICATION_CREDENTIALS'] = '../../BQ.json'

DB_info = {'server':'192.168.61.119:7622', 'user':'BAReporting', 'password':'KeHeCReme8he'}

client = bigquery.Client()

In [ ]:
sql = """
WITH target_users AS (
  SELECT
    'normal' AS user_type,
    '111.90.238.224' AS ip,
    '435bb410-6433-453a-95c0-590e78543a64' AS device_id,
    'openrice_ios/7.21.0 (iphone; ios 18.5)' AS user_agent

  UNION ALL SELECT
    'normal',
    '112.198.104.210',
    '72dd0c61450f9efd',
    'openrice_android/7.21.0 (android; 35) [cph2799]'

  UNION ALL SELECT
    'normal',
    '136.158.43.30',
    '8503e3f7ea531cf4',
    'openrice_android/7.21.0 (android; 36) [sm-a566b]'

  UNION ALL SELECT
    'abnormal',
    '223.123.7.182',
    'cc21b7620eaf4e65',
    'openrice_android/7.20.2 (android; 31)'

  UNION ALL SELECT
    'abnormal',
    '223.123.7.182',
    'c9a76abc2baaf940',
    'openrice_android/7.20.2 (android; 31)'

  UNION ALL SELECT
    'abnormal',
    '119.155.211.76',
    '907f26030cdd51e1',
    'openrice_android/7.20.4 (android; 33) [infinix x6836]'
)

SELECT
  target.user_type,
  pv.IP,
  pv.DeviceId,
  pv.SessionId,
  pv.Time,
  pv.EventAction,
  pv.EventLabel,
  pv.EventLabelRaw,
  pv.UserAgent
FROM `openrice-production.ORGA.PV_20260928` AS pv
INNER JOIN target_users AS target
  ON TRIM(CAST(pv.IP AS STRING)) = target.ip
  AND LOWER(pv.DeviceId) = target.device_id
  AND STARTS_WITH(LOWER(pv.UserAgent),target.user_agent)
  -- AND LOWER(pv.UserAgent) like '%' || target.user_agent || '%'
ORDER BY
  target.user_type,
  pv.IP,
  pv.DeviceId,
  pv.SessionId,
  pv.Time;
"""

In [23]:
df_bq = client.query(sql).result().to_dataframe()
df_bq

NotFound: 404 Not found: Table openrice-production:ORGA.PV_202609 was not found in location US; reason: notFound, message: Not found: Table openrice-production:ORGA.PV_202609 was not found in location US

Location: US
Job ID: 879e875a-7151-4b71-9fbf-5274cbf02f31
